# EDA — Virus detections (multi-source)

Exploratory load of the unified virus detections table (19-column schema from ENA, BV-BRC, Pathoplexus, FAO EMPRES-i, and ECDC NUTS3 feeds).

**Prerequisite:** run [`03_extract_viruses.ipynb`](03_extract_viruses.ipynb) first so `data/01_raw/virus/` contains `virus_detections.tsv` and `manifest.json`.

This notebook reads the dataset via `load_virus_detections`, inspects the multi-source extract manifest, per-`source` mix, pathogen / vector-group coverage (26 targets), Europe/Med vs worldwide scope (via `data/external/countries.csv`), `coord_source` / lat–lon / NUTS3 completeness, and temporal range. Deeper cleaning, H3 joins, and modelling features come later (see [`specs/03-spatial-temporal-matching-strategy.md`](../specs/03-spatial-temporal-matching-strategy.md)).


In [1]:
from __future__ import annotations

import json
import re
import pandas as pd

from vector_guard.data import (
    default_virus_raw_dir,
    find_repo_root,
    load_virus_detections,
)
from vector_guard.virus_extract import VIRUS_RAW_COLUMNS

REPO_ROOT = find_repo_root()
RAW_DIR = default_virus_raw_dir(REPO_ROOT)
MANIFEST_PATH = RAW_DIR / "manifest.json"
COUNTRIES_PATH = REPO_ROOT / "data" / "reference" / "countries.csv"

df = load_virus_detections()
assert list(df.columns) == list(VIRUS_RAW_COLUMNS), (
    f"Unexpected columns: {list(df.columns)}"
)
df.shape


(168277, 19)

## Source manifest

Metadata written by the extract notebook: fetch time, checksum, per-source / per-pathogen counts, citations, and `coord_source` tallies.


In [2]:
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print(f"fetched_at: {manifest.get('fetched_at')}")
    print(f"tsv_name:   {manifest.get('tsv_name')}")
    print(f"sha256:     {manifest.get('sha256')}")
    print(f"row_count:  {manifest.get('row_count'):,}")
    print(f"pre_dedupe: {manifest.get('pre_dedupe_row_count'):,}")
    print(f"columns:    {len(manifest.get('columns', []))}")
    print(f"filter:     {manifest.get('filter')}")
    print()
    print("per_source:")
    for source, n in sorted(
        (manifest.get("per_source") or {}).items(), key=lambda kv: -kv[1]
    ):
        print(f"  {source:16s} {n:,}")
    print()
    print("coord_source_counts:")
    for label, n in sorted(
        (manifest.get("coord_source_counts") or {}).items(), key=lambda kv: -kv[1]
    ):
        print(f"  {label:16s} {n:,}")
    print()
    print("citations:")
    for source, citation in (manifest.get("citations") or {}).items():
        print(f"  - {source}: {citation}")
else:
    print(f"No manifest at {MANIFEST_PATH}")
    manifest = None

{
    "dedupe_summary": (manifest or {}).get("dedupe_summary"),
    "per_vector_group": (manifest or {}).get("per_vector_group"),
    "n_targets": len((manifest or {}).get("targets") or []),
    "n_pathogens_in_manifest": len((manifest or {}).get("per_pathogen") or {}),
}


fetched_at: 2026-11-11T22:32:58.040003+00:00
tsv_name:   virus_detections.tsv
sha256:     c2020c8284784e1eada14cd84a19dcb337151859b0a1771c11f9f052c8ceb070
row_count:  168,277
pre_dedupe: 374,872
columns:    19
filter:     keep rows with non-empty country, nuts_id, (lat,lon), or collection_date; ENA sample backfill applied before keep filter; INSDC rows merged by normalized accession else sample_accession (+pathogen when multi-virus); outbreak rows deduplicated by native event_id or pathogen+loc+season

per_source:
  ena_sequence     118,381
  empres_i         25,874
  ena_read_run     13,231
  pathoplexus      6,377
  bvbrc            4,108
  ecdc_atlas       306

coord_source_counts:
  none             132,527
  exact_gps        34,187
  sample_backfill  1,563

citations:
  - ena_sequence: European Nucleotide Archive (ENA), EMBL-EBI. Sequence / read-run / sample metadata retrieved via the ENA Portal API (INSDC: ENA + GenBank + DDBJ). https://www.ebi.ac.uk/ena/portal/api/
  - ena_read_

{'dedupe_summary': {'pre_dedupe_total': 374872,
  'post_dedupe_total': 168277,
  'collapsed_rows': 206595,
  'insdc_pre_dedupe': 348475,
  'insdc_post_dedupe': 142097,
  'insdc_collapsed': 206378,
  'outbreak_pre_dedupe': 26397,
  'outbreak_post_dedupe': 26180,
  'outbreak_collapsed': 217,
  'pre_dedupe_by_source': {'bvbrc': 125334,
   'ena_sequence': 118384,
   'pathoplexus': 87646,
   'empres_i': 25874,
   'ena_read_run': 17111,
   'ecdc_atlas': 523},
  'post_dedupe_by_source': {'ena_sequence': 118381,
   'empres_i': 25874,
   'ena_read_run': 13231,
   'pathoplexus': 6377,
   'bvbrc': 4108,
   'ecdc_atlas': 306}},
 'per_vector_group': {'biting_midge': 42432,
  'mosquito': 110908,
  'sandfly': 707,
  'tick': 14230},
 'n_targets': 26,
 'n_pathogens_in_manifest': 26}

## First look

Unified 19-column schema: `record_id`, `source`, accessions, pathogen tags, place / NUTS / coords, `coord_source`, collection / host fields.


In [3]:
print(f"Rows: {len(df):,}")
print(f"Columns ({len(df.columns)}):")
df.dtypes


Rows: 168,277
Columns (19):


record_id             str
source                str
accession             str
sample_accession      str
study_accession       str
pathogen_name         str
vector_group          str
query_tax_id          str
scientific_name       str
tax_id                str
country               str
subnational_region    str
nuts_id               str
lat                   str
lon                   str
coord_source          str
collection_date       str
host                  str
isolation_source      str
dtype: object

In [4]:
df.head().T


,0,1,2,3,4
record_id,ena_sequence:AF206518,ena_sequence:AY187014,ena_sequence:AY278442,ena_sequence:AY428518,ena_sequence:AY428519
source,ena_sequence,ena_sequence,ena_sequence,ena_sequence,ena_sequence
accession,AF206518,AY187014,AY278442,AY428518,AY428519
sample_accession,,,,,
study_accession,,,,,
pathogen_name,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus
vector_group,mosquito,mosquito,mosquito,mosquito,mosquito
query_tax_id,3048448,3048448,3048448,3048448,3048448
scientific_name,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus
tax_id,11082,11082,11082,11082,11082


## Per-source breakdown

Row counts and place/time completeness by extract feed (`ena_sequence`, `ena_read_run`, `bvbrc`, `pathoplexus`, `empres_i`, `ecdc_atlas`).


In [5]:
def nonempty(series: pd.Series) -> pd.Series:
    return series.fillna("").astype(str).str.strip() != ""


by_source = (
    df["source"]
    .value_counts()
    .rename("rows")
    .to_frame()
    .assign(pct=lambda x: (100 * x["rows"] / len(df)).round(2))
)

source_completeness = (
    df.assign(
        has_country=nonempty(df["country"]),
        has_subnational=nonempty(df["subnational_region"]),
        has_nuts=nonempty(df["nuts_id"]),
        has_coords=nonempty(df["lat"]) & nonempty(df["lon"]),
        has_date=nonempty(df["collection_date"]),
        exact_gps=df["coord_source"].eq("exact_gps"),
        sample_backfill=df["coord_source"].eq("sample_backfill"),
    )
    .groupby("source", dropna=False)
    .agg(
        rows=("source", "size"),
        with_country=("has_country", "sum"),
        with_subnational=("has_subnational", "sum"),
        with_nuts_id=("has_nuts", "sum"),
        with_lat_lon=("has_coords", "sum"),
        exact_gps=("exact_gps", "sum"),
        sample_backfill=("sample_backfill", "sum"),
        with_date=("has_date", "sum"),
    )
    .assign(
        lat_lon_pct=lambda x: (100 * x["with_lat_lon"] / x["rows"]).round(2),
        nuts_pct=lambda x: (100 * x["with_nuts_id"] / x["rows"]).round(2),
    )
    .sort_values("rows", ascending=False)
)

print("Rows by source:")
display(by_source)
print("Completeness by source:")
source_completeness


Rows by source:


,rows,pct
source,,
ena_sequence,118381,70.35
empres_i,25874,15.38
ena_read_run,13231,7.86
pathoplexus,6377,3.79
bvbrc,4108,2.44
ecdc_atlas,306,0.18


Completeness by source:


,rows,with_country,with_subnational,with_nuts_id,with_lat_lon,exact_gps,sample_backfill,with_date,lat_lon_pct,nuts_pct
source,,,,,,,,,,
ena_sequence,118381,117822,48833,0,5482,3919,1563,111189,4.63,0.0
empres_i,25874,25874,25874,0,25874,25874,0,25874,100.00,0.0
ena_read_run,13231,12573,6368,0,4250,4250,0,13204,32.12,0.0
pathoplexus,6377,6377,3790,0,144,144,0,6068,2.26,0.0
bvbrc,4108,3993,1721,0,0,0,0,3930,0.00,0.0
ecdc_atlas,306,306,306,306,0,0,0,306,0.00,100.0


## Pathogen and vector-group distribution

Coverage across the 26-pathogen `VIRUS_VECTOR_TAXON_MAP` target set and four VectorNet vector groups.


In [6]:
by_pathogen = (
    df["pathogen_name"]
    .value_counts()
    .rename("rows")
    .to_frame()
    .assign(pct=lambda x: (100 * x["rows"] / len(df)).round(2))
)
by_vector_group = (
    df["vector_group"]
    .value_counts()
    .rename("rows")
    .to_frame()
    .assign(pct=lambda x: (100 * x["rows"] / len(df)).round(2))
)

pathogen_by_source = (
    df.groupby(["pathogen_name", "source"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=sorted(df["source"].unique()), fill_value=0)
)
pathogen_by_source["total"] = pathogen_by_source.sum(axis=1)
pathogen_by_source = pathogen_by_source.sort_values("total", ascending=False)

print(f"Distinct pathogens: {df['pathogen_name'].nunique()}")
print("By pathogen_name:")
display(by_pathogen)
print("By vector_group:")
display(by_vector_group)
print("Pathogen × source:")
pathogen_by_source


Distinct pathogens: 26
By pathogen_name:


,rows,pct
pathogen_name,,
Dengue virus,66420,39.47
Bluetongue virus,32578,19.36
West Nile virus,14424,8.57
Chikungunya virus,12177,7.24
Crimean-Congo hemorrhagic fever virus,8511,5.06
Zika virus,4722,2.81
Schmallenberg virus,3959,2.35
Japanese encephalitis virus,3768,2.24
Rift Valley fever virus,3737,2.22


By vector_group:


,rows,pct
vector_group,,
mosquito,110908,65.91
biting_midge,42432,25.22
tick,14230,8.46
sandfly,707,0.42


Pathogen × source:


source,bvbrc,ecdc_atlas,empres_i,ena_read_run,ena_sequence,pathoplexus,total
pathogen_name,,,,,,,
Dengue virus,1184,0,0,4918,57253,3065,66420
Bluetongue virus,298,0,18120,188,13972,0,32578
West Nile virus,50,306,2172,3158,8519,219,14424
Chikungunya virus,972,0,0,1145,10030,30,12177
Crimean-Congo hemorrhagic fever virus,106,0,0,103,5272,3030,8511
Zika virus,30,0,0,2416,2245,31,4722
Schmallenberg virus,9,0,3349,6,595,0,3959
Japanese encephalitis virus,44,0,190,138,3396,0,3768
Rift Valley fever virus,37,0,1803,137,1760,0,3737


## Geographic coverage — Europe vs worldwide

Country tokens are matched to [`data/external/countries.csv`](../data/external/countries.csv) (canonical name + `name_aliases`); `europe_med` flags the Europe / Mediterranean modelling scope.


In [7]:
countries = pd.read_csv(COUNTRIES_PATH, dtype=str).fillna("")
countries["europe_med"] = countries["europe_med"].astype(int)


def build_country_lookup(frame: pd.DataFrame) -> dict[str, dict[str, str | int]]:
    lookup: dict[str, dict[str, str | int]] = {}
    for row in frame.itertuples(index=False):
        meta = {
            "iso2": row.iso2,
            "name": row.name,
            "europe_med": int(row.europe_med),
        }
        tokens = {row.name, row.iso2}
        for alias in str(row.name_aliases).split("|"):
            alias = alias.strip()
            if alias:
                tokens.add(alias)
        for token in tokens:
            lookup[token.casefold()] = meta
    return lookup


COUNTRY_LOOKUP = build_country_lookup(countries)


def parse_country_token(value: str) -> str:
    text = (value or "").strip()
    if not text:
        return ""
    return text.split(":", 1)[0].strip()


def resolve_country(token: str) -> dict[str, str | int]:
    if not token:
        return {"iso2": "", "name": "", "europe_med": 0}
    hit = COUNTRY_LOOKUP.get(token.casefold())
    if hit is None:
        return {"iso2": "", "name": token, "europe_med": 0}
    return hit


geo = df.copy()
geo["country_token"] = geo["country"].map(parse_country_token)
resolved = geo["country_token"].map(resolve_country)
geo["country_iso2"] = resolved.map(lambda m: m["iso2"])
geo["country_name"] = resolved.map(lambda m: m["name"])
geo["mapped"] = geo["country_iso2"].ne("")
geo["region_scope"] = [
    (
        "missing"
        if not token
        else "europe_med"
        if bool(meta["europe_med"])
        else "worldwide_other"
    )
    for token, meta in zip(geo["country_token"], resolved, strict=True)
]

scope_summary = (
    geo["region_scope"]
    .value_counts()
    .rename("rows")
    .to_frame()
    .assign(pct=lambda x: (100 * x["rows"] / len(geo)).round(2))
)
unmapped = (
    geo.loc[~geo["mapped"] & geo["country_token"].ne(""), "country_token"]
    .value_counts()
    .head(20)
    .to_frame("rows")
)

print("Region scope (Europe/Med vs other vs missing country):")
display(scope_summary)
print(f"\nMapped to countries.csv: {int(geo['mapped'].sum()):,} "
      f"({100 * geo['mapped'].mean():.1f}%)")
print("\nTop countries overall (resolved name or raw token):")
display(
    geo.assign(
        label=lambda x: x["country_name"].where(x["country_name"].ne(""), "(missing)")
    )["label"]
    .value_counts()
    .head(20)
    .to_frame("rows")
)
print("\nTop Europe / Mediterranean countries:")
display(
    geo.loc[geo["region_scope"] == "europe_med", "country_name"]
    .value_counts()
    .head(20)
    .to_frame("rows")
)
print("\nTop unmapped country tokens (non-ISO aliases / regions / oceans):")
unmapped


Region scope (Europe/Med vs other vs missing country):


,rows,pct
region_scope,,
worldwide_other,119131,70.79
europe_med,47814,28.41
missing,1332,0.79



Mapped to countries.csv: 164,421 (97.7%)

Top countries overall (resolved name or raw token):


,rows
label,
United States,14562
Brazil,13840
China,10513
India,10225
France,8440
Thailand,7417
Russian Federation,6058
South Africa,5626
Vietnam,5159



Top Europe / Mediterranean countries:


,rows
country_name,
France,8440
Russian Federation,6058
Italy,4808
Greece,4483
Germany,3408
Türkiye,2145
Israel,1545
Bulgaria,1496
Romania,1447



Top unmapped country tokens (non-ISO aliases / regions / oceans):


,rows
country_token,
the United Kingdom of Great Britain and Northern Ireland,1278
Netherlands (Kingdom of the),404
Namibia,264
missing,202
Not Applicable,182
Not applicable,89
Czechoslovakia,22
Swaziland,17
505 [Nicaragua],16


In [8]:
scope_by_pathogen = (
    geo.groupby(["pathogen_name", "region_scope"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["europe_med", "worldwide_other", "missing"], fill_value=0)
)
scope_by_pathogen["total"] = scope_by_pathogen.sum(axis=1)
scope_by_pathogen["europe_med_pct"] = (
    100 * scope_by_pathogen["europe_med"] / scope_by_pathogen["total"]
).round(2)

scope_by_source = (
    geo.groupby(["source", "region_scope"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["europe_med", "worldwide_other", "missing"], fill_value=0)
)
scope_by_source["total"] = scope_by_source.sum(axis=1)
scope_by_source["europe_med_pct"] = (
    100 * scope_by_source["europe_med"] / scope_by_source["total"]
).round(2)

print("Region scope by pathogen:")
display(scope_by_pathogen.sort_values("total", ascending=False))
print("Region scope by source:")
scope_by_source.sort_values("total", ascending=False)


Region scope by pathogen:


region_scope,europe_med,worldwide_other,missing,total,europe_med_pct
pathogen_name,,,,,
Dengue virus,582,65539,299,66420,0.88
Bluetongue virus,24080,8349,149,32578,73.91
West Nile virus,5536,8670,218,14424,38.38
Chikungunya virus,141,11984,52,12177,1.16
Crimean-Congo hemorrhagic fever virus,5786,2700,25,8511,67.98
Zika virus,337,4163,222,4722,7.14
Schmallenberg virus,3266,692,1,3959,82.50
Japanese encephalitis virus,2,3752,14,3768,0.05
Rift Valley fever virus,106,3629,2,3737,2.84


Region scope by source:


region_scope,europe_med,worldwide_other,missing,total,europe_med_pct
source,,,,,
ena_sequence,21814,96008,559,118381,18.43
empres_i,21869,4005,0,25874,84.52
ena_read_run,1135,11438,658,13231,8.58
pathoplexus,2232,4145,0,6377,35.00
bvbrc,462,3531,115,4108,11.25
ecdc_atlas,302,4,0,306,98.69


## Coordinate, NUTS3, and region completeness

H3 Tier-1 joins prefer exact GPS (`coord_source` in `{exact_gps, sample_backfill}`) or NUTS3 codes (`nuts_id`, mainly ECDC). Rows with only country / `subnational_region` text fall back toward country Tier 2 in the matching strategy.


In [9]:
geo["has_country"] = geo["country_token"].ne("")
geo["has_subnational"] = nonempty(geo["subnational_region"])
geo["has_nuts_id"] = nonempty(geo["nuts_id"])
geo["has_lat_lon"] = nonempty(geo["lat"]) & nonempty(geo["lon"])
geo["coord_tier"] = geo["coord_source"].where(
    geo["coord_source"].isin(["exact_gps", "sample_backfill", "none"]),
    other="none",
)
# Treat lat/lon presence as the operational GPS flag even if label drifted.
geo.loc[geo["has_lat_lon"] & geo["coord_tier"].eq("none"), "coord_tier"] = "exact_gps"

completeness = pd.Series(
    {
        "has_country": int(geo["has_country"].sum()),
        "has_subnational_region": int(geo["has_subnational"].sum()),
        "has_nuts_id": int(geo["has_nuts_id"].sum()),
        "has_lat_lon": int(geo["has_lat_lon"].sum()),
        "coord_exact_gps": int(geo["coord_source"].eq("exact_gps").sum()),
        "coord_sample_backfill": int(geo["coord_source"].eq("sample_backfill").sum()),
        "coord_none": int(geo["coord_source"].eq("none").sum()),
        "tier1_gps_or_nuts": int((geo["has_lat_lon"] | geo["has_nuts_id"]).sum()),
    },
    name="rows",
).to_frame()
completeness["pct"] = (100 * completeness["rows"] / len(geo)).round(2)

geo["tier1_ready"] = geo["has_lat_lon"] | geo["has_nuts_id"]
coord_by_scope = (
    geo.groupby("region_scope")
    .agg(
        rows=("region_scope", "size"),
        with_lat_lon=("has_lat_lon", "sum"),
        with_nuts_id=("has_nuts_id", "sum"),
        tier1_ready=("tier1_ready", "sum"),
        exact_gps=("coord_source", lambda s: int(s.eq("exact_gps").sum())),
        sample_backfill=("coord_source", lambda s: int(s.eq("sample_backfill").sum())),
    )
    .assign(
        lat_lon_pct=lambda x: (100 * x["with_lat_lon"] / x["rows"]).round(2),
        nuts_pct=lambda x: (100 * x["with_nuts_id"] / x["rows"]).round(2),
        tier1_pct=lambda x: (100 * x["tier1_ready"] / x["rows"]).round(2),
    )
)

coord_by_source = (
    geo.groupby("source")["coord_source"]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=["exact_gps", "sample_backfill", "none"], fill_value=0)
)

europe = geo.loc[geo["region_scope"] == "europe_med"].copy()
europe_tier1 = europe.loc[europe["has_lat_lon"] | europe["has_nuts_id"]]

print("Field completeness (all rows):")
display(completeness)
print("\nCoord / NUTS coverage by region scope:")
display(coord_by_scope)
print("\ncoord_source × source:")
display(coord_by_source)
print(
    f"\nEurope/Med Tier-1 candidates (lat/lon or nuts_id): "
    f"{len(europe_tier1):,} / {len(europe):,} "
    f"({100 * len(europe_tier1) / max(len(europe), 1):.1f}%)"
)
print("\nEurope/Med Tier-1 by source:")
display(
    europe_tier1["source"]
    .value_counts()
    .rename("rows")
    .to_frame()
    .assign(pct=lambda x: (100 * x["rows"] / len(europe_tier1)).round(2))
)
print("\nNUTS3 rows (nonzero nuts_id) by source / pathogen:")
(
    geo.loc[geo["has_nuts_id"]]
    .groupby(["source", "pathogen_name"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)


Field completeness (all rows):


,rows,pct
has_country,166945,99.21
has_subnational_region,86892,51.64
has_nuts_id,306,0.18
has_lat_lon,35750,21.24
coord_exact_gps,34187,20.32
coord_sample_backfill,1563,0.93
coord_none,132527,78.76
tier1_gps_or_nuts,36056,21.43



Coord / NUTS coverage by region scope:


,rows,with_lat_lon,with_nuts_id,tier1_ready,exact_gps,sample_backfill,lat_lon_pct,nuts_pct,tier1_pct
region_scope,,,,,,,,,
europe_med,47814,23311,302,23613,23040,271,48.75,0.63,49.39
missing,1332,124,0,124,124,0,9.31,0.00,9.31
worldwide_other,119131,12315,4,12319,11023,1292,10.34,0.00,10.34



coord_source × source:


coord_source,exact_gps,sample_backfill,none
source,,,
bvbrc,0,0,4108
ecdc_atlas,0,0,306
empres_i,25874,0,0
ena_read_run,4250,0,8981
ena_sequence,3919,1563,112899
pathoplexus,144,0,6233



Europe/Med Tier-1 candidates (lat/lon or nuts_id): 23,613 / 47,814 (49.4%)

Europe/Med Tier-1 by source:


,rows,pct
source,,
empres_i,21869,92.61
ena_read_run,826,3.50
ena_sequence,616,2.61
ecdc_atlas,302,1.28



NUTS3 rows (nonzero nuts_id) by source / pathogen:


,source,pathogen_name,rows
0,ecdc_atlas,West Nile virus,306


## Temporal range

Parse a coarse year from `collection_date` (formats vary: `2003`, `02-Aug-2005`, ISO dates, ranges, etc.).


In [10]:
def parse_year(value: str) -> int | None:
    text = (value or "").strip()
    if not text:
        return None
    match = re.search(r"(19|20)\d{2}", text)
    if not match:
        return None
    year = int(match.group(0))
    if 1900 <= year <= 2100:
        return year
    return None


geo["year"] = geo["collection_date"].map(parse_year)
with_year = int(geo["year"].notna().sum())
print(f"Rows with parseable year: {with_year:,} ({100 * with_year / len(geo):.1f}%)")

years = geo["year"].dropna().astype(int)
if len(years):
    print(f"Year range: {years.min()} – {years.max()}")
    print("\nRows by decade:")
    decade = (years // 10 * 10).value_counts().sort_index()
    display(decade.to_frame("rows"))

print("\nYear coverage by pathogen (min / max / n):")
display(
    geo.dropna(subset=["year"])
    .groupby("pathogen_name")["year"]
    .agg(min_year="min", max_year="max", n="count")
    .astype({"min_year": int, "max_year": int})
    .sort_values("n", ascending=False)
)

print("\nYear coverage by source (min / max / n):")
(
    geo.dropna(subset=["year"])
    .groupby("source")["year"]
    .agg(min_year="min", max_year="max", n="count")
    .astype({"min_year": int, "max_year": int})
    .sort_values("n", ascending=False)
)


Rows with parseable year: 158,578 (94.2%)


Year range: 1900 – 2026

Rows by decade:


,rows
year,
1900,13
1920,2
1930,75
1940,217
1950,487
1960,1063
1970,975
1980,2250
1990,3743



Year coverage by pathogen (min / max / n):


,min_year,max_year,n
pathogen_name,,,
Dengue virus,1905,2026,63383
Bluetongue virus,1900,2026,30786
West Nile virus,1931,2026,13748
Chikungunya virus,1952,2026,11691
Crimean-Congo hemorrhagic fever virus,1956,2025,7950
Schmallenberg virus,1965,2026,3933
Zika virus,1947,2026,3872
Rift Valley fever virus,1944,2026,3675
Japanese encephalitis virus,1935,2026,3214



Year coverage by source (min / max / n):


,min_year,max_year,n
source,,,
ena_sequence,1905,2026,110592
empres_i,2000,2026,25874
ena_read_run,1900,2026,11810
pathoplexus,1956,2026,6068
bvbrc,1900,2025,3928
ecdc_atlas,2026,2026,306


## Quick takeaways

- The extract is a **multi-source** unified table (INSDC sequence/run + curated portals + outbreak/admin feeds), not ENA-only.
- Pathogen coverage spans the **26-target** taxonomy; mix remains imbalanced (dengue / bluetongue dominate globally).
- Europe/Med is a minority of worldwide rows; outbreak feeds (`empres_i`, `ecdc_atlas`) and sample backfill improve local densification.
- Tier 1 readiness: use `coord_source` in `{exact_gps, sample_backfill}` for GPS points and `nuts_id` (ECDC) for NUTS3 centroids; remaining rows stay at country / subnational Tier 2.
- Next steps: refresh country / subnational reference tables for new source tokens, then H3 assignment and VectorNet joins per the matching strategy spec.
